# Training & Evaluation Corpus: Descriptive Tables

## 1. Setup & Data Loading

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CODE_DIR = Path("../..").resolve()
for p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(CODE_DIR / "analysis" / "wang2026")):
    if p not in sys.path:
        sys.path.insert(0, p)

from data.base import ALL_PERSONA_NAMES, SINGLE_PERSONA_ATTRS
from data.grummon import GRUMMON_PARQUET_PATH, GRUMMON_PERSONA_COLS
from data.issp import ISSP_YEARS
from data.persona import (
    DATASET_PERSONA_COLS,
    GLOBAL_PERSONA_LEVELS,
    apply_persona_mapping,
)
from wang_mnl import load_covid_conjoint, pair_choice_sets

DATA_DIR = CODE_DIR / "data"

VALID_ISSP_QIDS = ["2020:v12", "2020:v6", "2020:v28", "2020:v42"]
VALID_TWIN_QIDS = [f"twin2k:QID9_{i:03d}" for i in range(89, 100)] + ["twin2k:QID41", "twin2k:QID163", "twin2k:QID165"]
VALID_QIDS = VALID_ISSP_QIDS + VALID_TWIN_QIDS

issp_frames = []
for y in ISSP_YEARS:
    df = pd.read_parquet(DATA_DIR / f"issp{y}" / f"issp{y}_US_long.parquet")
    df = apply_persona_mapping(df, dataset="issp")
    df["year"] = y
    df["qid"] = y + ":" + df["v"]
    df["dataset"] = "US ISSP (2015 to 2024)"
    df["resp_uid"] = "issp:" + df["CASEID"].astype(str)
    issp_frames.append(df)
issp_df = pd.concat(issp_frames, ignore_index=True)

twin_df = pd.read_parquet(DATA_DIR / "Twin-2K-500" / "twin2k_US_long.parquet")
twin_df["year"] = "twin2k"
twin_df["qid"] = "twin2k:" + twin_df["v"]
twin_df["dataset"] = "Twin-2K-500"
twin_df["resp_uid"] = "twin2k:" + twin_df["CASEID"].astype(str)

all_df = pd.concat([issp_df, twin_df], ignore_index=True)
if not set(VALID_QIDS) <= set(all_df["qid"].unique()):
    raise KeyError(f"Missing VALID_QIDS in all_df: {sorted(set(VALID_QIDS) - set(all_df['qid'].unique()))}")

all_df["split"] = np.where(all_df["qid"].isin(VALID_QIDS), "Validation", "Training")

covid_raw = load_covid_conjoint(DATA_DIR / "covid" / "dataverse_files" / "Kreps_etal_vax_replication_data.dta")
covid_df_a, _, _, _, _ = pair_choice_sets(covid_raw)
grummon_df = pd.read_parquet(GRUMMON_PARQUET_PATH)

print(
    f"Loaded {len(all_df):,} survey observations ({all_df['resp_uid'].nunique():,} respondents, {all_df['qid'].nunique():,} questions), "
    f"{len(covid_df_a):,} COVID-19 conjoint choice sets, and {len(grummon_df):,} nutrition RCT choices."
)

Loaded 1,895,133 survey observations (16,414 respondents, 1,590 questions), 9,855 COVID-19 conjoint choice sets, and 33,798 nutrition RCT choices.


## 2. Main-Text Table: Summary of Datasets (`#tbl-datasets`)

In [2]:
def summarize_survey_slice(sub_df: pd.DataFrame, split_label: str, dataset_label: str, n_attrs: int) -> dict:
    q_meta = sub_df.groupby("qid", as_index=False).agg(q_type=("type", "first"))
    n_q = len(q_meta)
    n_score = int((q_meta["q_type"] == "score").sum())
    n_choice = int((q_meta["q_type"] == "choice").sum())
    if n_score + n_choice != n_q:
        raise ValueError(f"Unexpected question types in {split_label} / {dataset_label}")
    n_resp = int(sub_df["resp_uid"].nunique())
    return {
        "Split": split_label,
        "Dataset": dataset_label,
        "Persona attributes": n_attrs,
        "Questions / tasks": f"{n_q:,}",
        "Likert (score)": f"{n_score:,}",
        "Categorical (choice)": f"{n_choice:,}",
        "Unique respondents": f"{n_resp:,}",
        "Mean items / respondent": f"{len(sub_df) / n_resp:.1f}",
        "Individual observations": f"{len(sub_df):,}",
    }


n_issp_attrs = len([c for c in DATASET_PERSONA_COLS["issp"] if c != "c_alphan"])
n_twin_attrs = len([c for c in DATASET_PERSONA_COLS["twin2k"] if c != "c_alphan"])
n_comb_attrs = len(SINGLE_PERSONA_ATTRS)
n_covid_attrs = len([c for c in DATASET_PERSONA_COLS["covid"] if c != "c_alphan"])
n_grummon_attrs = len(GRUMMON_PERSONA_COLS)

summary_rows = []
for split_label in ("Training", "Validation"):
    sp_df = all_df[all_df["split"] == split_label]
    summary_rows.append(summarize_survey_slice(sp_df[sp_df["dataset"] == "US ISSP (2015 to 2024)"], split_label, "US ISSP (2015 to 2024)", n_issp_attrs))
    summary_rows.append(summarize_survey_slice(sp_df[sp_df["dataset"] == "Twin-2K-500"], split_label, "Twin-2K-500", n_twin_attrs))
    summary_rows.append(summarize_survey_slice(sp_df, split_label, "Combined", n_comb_attrs))

# Out-of-domain marketing evaluation benchmarks
n_covid_resp = int(covid_df_a["respondent"].nunique())
n_covid_pairs = int(len(covid_df_a))
n_grummon_resp = int(grummon_df["CASEID"].nunique())
n_grummon_q = int(grummon_df["qid"].nunique())

summary_rows.extend([
    {
        "Split": "Out-of-domain test",
        "Dataset": "COVID-19 vaccine conjoint (full / eval subsample)",
        "Persona attributes": n_covid_attrs,
        "Questions / tasks": "5 / 5",
        "Likert (score)": "0",
        "Categorical (choice)": "5 / 5",
        "Unique respondents": f"{n_covid_resp:,} / 200",
        "Mean items / respondent": "5.0",
        "Individual observations": f"{n_covid_pairs:,} / 1,000",
    },
    {
        "Split": "Out-of-domain test",
        "Dataset": "Nutrition labeling RCT (6 arms x 6 categories)",
        "Persona attributes": n_grummon_attrs,
        "Questions / tasks": f"{n_grummon_q:,}",
        "Likert (score)": "0",
        "Categorical (choice)": f"{n_grummon_q:,}",
        "Unique respondents": f"{n_grummon_resp:,}",
        "Mean items / respondent": f"{len(grummon_df) / n_grummon_resp:.1f}",
        "Individual observations": f"{len(grummon_df):,}",
    },
])

corpus_summary_df = pd.DataFrame(summary_rows)
display(corpus_summary_df)

md_lines = [
    "| Split | Dataset | Attributes | Questions | Likert (`score`) | Categorical (`choice`) | Respondents | Observations |",
    "|:---|:---|---:|---:|---:|---:|---:|---:|",
]
for r in summary_rows:
    md_lines.append(
        f"| {r['Split']} | {r['Dataset']} | {r['Persona attributes']} | {r['Questions / tasks']} | "
        f"{r['Likert (score)']} | {r['Categorical (choice)']} | {r['Unique respondents']} | {r['Individual observations']} |"
    )
md_lines.append("\n: Summary of the respondent-level training corpus, held-out validation split, and out-of-domain marketing evaluation benchmarks. {#tbl-datasets}")
print("\n".join(md_lines))

,Split,Dataset,Persona attributes,Questions / tasks,Likert (score),Categorical (choice),Unique respondents,Mean items / respondent,Individual observations
0,Training,US ISSP (2015 to 2024),15,612,470,142,"14,356",53.0,"760,473"
1,Training,Twin-2K-500,13,960,287,673,"2,058",544.9,"1,121,477"
2,Training,Combined,17,"1,572",757,815,"16,414",114.7,"1,881,950"
3,Validation,US ISSP (2015 to 2024),15,4,4,0,"1,827",3.8,"7,011"
4,Validation,Twin-2K-500,13,14,0,14,"2,058",3.0,"6,172"
5,Validation,Combined,17,18,4,14,"3,885",3.4,"13,183"
6,Out-of-domain test,COVID-19 vaccine conjoint (full / eval subsample),8,5 / 5,0,5 / 5,"1,971 / 200",5.0,"9,855 / 1,000"
7,Out-of-domain test,Nutrition labeling RCT (6 arms x 6 categories),10,36,0,36,"5,633",6.0,"33,798"


| Split | Dataset | Attributes | Questions | Likert (`score`) | Categorical (`choice`) | Respondents | Observations |
|:---|:---|---:|---:|---:|---:|---:|---:|
| Training | US ISSP (2015 to 2024) | 15 | 612 | 470 | 142 | 14,356 | 760,473 |
| Training | Twin-2K-500 | 13 | 960 | 287 | 673 | 2,058 | 1,121,477 |
| Training | Combined | 17 | 1,572 | 757 | 815 | 16,414 | 1,881,950 |
| Validation | US ISSP (2015 to 2024) | 15 | 4 | 4 | 0 | 1,827 | 7,011 |
| Validation | Twin-2K-500 | 13 | 14 | 0 | 14 | 2,058 | 6,172 |
| Validation | Combined | 17 | 18 | 4 | 14 | 3,885 | 13,183 |
| Out-of-domain test | COVID-19 vaccine conjoint (full / eval subsample) | 8 | 5 / 5 | 0 | 5 / 5 | 1,971 / 200 | 9,855 / 1,000 |
| Out-of-domain test | Nutrition labeling RCT (6 arms x 6 categories) | 10 | 36 | 0 | 36 | 5,633 | 33,798 |

: Summary of the respondent-level training corpus, held-out validation split, and out-of-domain marketing evaluation benchmarks. {#tbl-datasets}


## 3. Web Appendix Table: US ISSP Waves (`#apptbl-issp-waves`)

In [3]:
ISSP_WAVE_TOPICS = {
    "2015": "Work orientations IV",
    "2016": "Role of government V",
    "2017": "Social networks and social resources",
    "2018": "Religion IV",
    "2019": "Social inequality V",
    "2020": "Environment IV",
    "2021": "Health and health care II",
    "2022": "Family and changing gender roles V",
    "2023": "National identity and citizenship",
    "2024": "Digital societies",
}

wave_rows = []
for y in ISSP_YEARS:
    w_df = issp_df[issp_df["year"] == y]
    q_meta = w_df.groupby("qid", as_index=False).agg(q_type=("type", "first"))
    resp_counts = w_df.groupby("resp_uid").size()
    wave_rows.append({
        "Wave": y,
        "Thematic module": ISSP_WAVE_TOPICS[y],
        "Total questions": len(q_meta),
        "Likert (score)": int((q_meta["q_type"] == "score").sum()),
        "Categorical (choice)": int((q_meta["q_type"] == "choice").sum()),
        "Unique respondents": int(w_df["resp_uid"].nunique()),
        "Mean Qs / respondent": round(float(resp_counts.mean()), 1),
        "Total responses": int(len(w_df)),
    })

issp_wave_df = pd.DataFrame(wave_rows)
display(issp_wave_df)

wave_md = [
    "| Wave | Thematic module | Questions | Likert (`score`) | Categorical (`choice`) | Respondents | Mean Qs / respondent | Responses |",
    "|:---|:---|---:|---:|---:|---:|---:|---:|",
]
for r in wave_rows:
    wave_md.append(
        f"| {r['Wave']} | {r['Thematic module']} | {r['Total questions']:,} | {r['Likert (score)']:,} | "
        f"{r['Categorical (choice)']:,} | {r['Unique respondents']:,} | {r['Mean Qs / respondent']:.1f} | {r['Total responses']:,} |"
    )
wave_md.append(
    f"| Total | All 10 US waves (2015 to 2024) | {sum(r['Total questions'] for r in wave_rows):,} | "
    f"{sum(r['Likert (score)'] for r in wave_rows):,} | {sum(r['Categorical (choice)'] for r in wave_rows):,} | "
    f"{issp_df['resp_uid'].nunique():,} | {len(issp_df) / issp_df['resp_uid'].nunique():.1f} | {len(issp_df):,} |"
)
wave_md.append("\n: Summary of the 10 US waves of the International Social Survey Programme (ISSP 2015 to 2024). {#apptbl-issp-waves}")
print("\n".join(wave_md))

,Wave,Thematic module,Total questions,Likert (score),Categorical (choice),Unique respondents,Mean Qs / respondent,Total responses
0,2015,Work orientations IV,82,68,14,1474,49.1,72416
1,2016,Role of government V,60,52,8,1384,58.6,81043
2,2017,Social networks and social resources,60,32,28,1173,59.2,69500
3,2018,Religion IV,62,50,12,1175,60.2,70719
4,2019,Social inequality V,49,44,5,1852,45.9,84963
5,2020,Environment IV,55,45,10,1847,52.3,96598
6,2021,Health and health care II,58,50,8,1146,54.8,62793
7,2022,Family and changing gender roles V,55,32,23,1126,45.6,51303
8,2023,National identity and citizenship,60,56,4,1589,56.3,89391
9,2024,Digital societies,75,45,30,1590,55.8,88758


| Wave | Thematic module | Questions | Likert (`score`) | Categorical (`choice`) | Respondents | Mean Qs / respondent | Responses |
|:---|:---|---:|---:|---:|---:|---:|---:|
| 2015 | Work orientations IV | 82 | 68 | 14 | 1,474 | 49.1 | 72,416 |
| 2016 | Role of government V | 60 | 52 | 8 | 1,384 | 58.6 | 81,043 |
| 2017 | Social networks and social resources | 60 | 32 | 28 | 1,173 | 59.2 | 69,500 |
| 2018 | Religion IV | 62 | 50 | 12 | 1,175 | 60.2 | 70,719 |
| 2019 | Social inequality V | 49 | 44 | 5 | 1,852 | 45.9 | 84,963 |
| 2020 | Environment IV | 55 | 45 | 10 | 1,847 | 52.3 | 96,598 |
| 2021 | Health and health care II | 58 | 50 | 8 | 1,146 | 54.8 | 62,793 |
| 2022 | Family and changing gender roles V | 55 | 32 | 23 | 1,126 | 45.6 | 51,303 |
| 2023 | National identity and citizenship | 60 | 56 | 4 | 1,589 | 56.3 | 89,391 |
| 2024 | Digital societies | 75 | 45 | 30 | 1,590 | 55.8 | 88,758 |
| Total | All 10 US waves (2015 to 2024) | 616 | 474 | 142 | 14,356 | 53.5 | 767,484 |

: S

## 4. Web Appendix Table: Harmonized Persona Attributes (`#apptbl-attributes`)

In [4]:
grummon_resp_df = grummon_df.groupby("CASEID", as_index=False).first()
issp_set = set(DATASET_PERSONA_COLS["issp"]) - {"c_alphan"}
twin_set = set(DATASET_PERSONA_COLS["twin2k"]) - {"c_alphan"}
covid_set = set(DATASET_PERSONA_COLS["covid"]) - {"c_alphan"}
grummon_set = set(GRUMMON_PERSONA_COLS)

all_attrs = list(SINGLE_PERSONA_ATTRS) + [c for c in GRUMMON_PERSONA_COLS if c not in SINGLE_PERSONA_ATTRS]
attr_rows = []
for col in all_attrs:
    if col in GLOBAL_PERSONA_LEVELS:
        levels = [str(x) for x in GLOBAL_PERSONA_LEVELS[col]]
    else:
        levels = [str(x) for x in sorted(grummon_resp_df[col].dropna().unique().tolist())]

    if len(levels) <= 6:
        level_summary = f"{len(levels)} ({", ".join(levels)})"
    else:
        level_summary = f"{len(levels)} ({", ".join(levels[:3])}, ..., {levels[-1]})"

    datasets = []
    if col in issp_set:
        datasets.append("ISSP")
    if col in twin_set:
        datasets.append("Twin-2K")
    if col in covid_set:
        datasets.append("Conjoint")
    if col in grummon_set:
        datasets.append("Nutrition RCT")

    attr_rows.append({
        "Prompt attribute": ALL_PERSONA_NAMES[col],
        "Harmonized levels": level_summary.replace("$", "\\$"),
        "Datasets": ", ".join(datasets),
    })

attr_df = pd.DataFrame(attr_rows)
display(attr_df)

attr_md = [
    "| Prompt attribute | Harmonized levels | Datasets |",
    "|:---|:---|:---|",
]
for r in attr_rows:
    attr_md.append(f"| {r['Prompt attribute']} | {r['Harmonized levels']} | {r['Datasets']} |")
attr_md.append("\n: Harmonized respondent persona attributes, category levels, and dataset inclusion across training and evaluation benchmarks. {#apptbl-attributes}")
print("\n".join(attr_md))

,Prompt attribute,Harmonized levels,Datasets
0,Sex,"2 (Male, Female)","ISSP, Twin-2K, Conjoint, Nutrition RCT"
1,Age,"4 (18-29, 30-49, 50-64, 65+)","ISSP, Twin-2K, Conjoint, Nutrition RCT"
2,Education,"4 (Less than high school, High school or some ...","ISSP, Twin-2K, Conjoint, Nutrition RCT"
3,Religion,"12 (Protestant, Catholic, Orthodox, ..., Other)","ISSP, Twin-2K, Conjoint"
4,Attendance of religious services,"6 (More than once a week, Once a week, Once or...","ISSP, Twin-2K"
5,Political ideology,"8 (Very liberal, Liberal, Somewhat liberal, .....","ISSP, Twin-2K, Conjoint"
6,Voted in the last general election,"2 (Yes, No)",ISSP
7,Place of living,"5 (A big city, The suburbs or outskirts of a b...",ISSP
8,Marital status,"6 (Married, Living with a partner, Divorced, S...","ISSP, Twin-2K"
9,Main activity,"10 (Full-time employment, Part-time employment...","ISSP, Twin-2K"


| Prompt attribute | Harmonized levels | Datasets |
|:---|:---|:---|
| Sex | 2 (Male, Female) | ISSP, Twin-2K, Conjoint, Nutrition RCT |
| Age | 4 (18-29, 30-49, 50-64, 65+) | ISSP, Twin-2K, Conjoint, Nutrition RCT |
| Education | 4 (Less than high school, High school or some college, Associate's degree, College graduate/some postgrad) | ISSP, Twin-2K, Conjoint, Nutrition RCT |
| Religion | 12 (Protestant, Catholic, Orthodox, ..., Other) | ISSP, Twin-2K, Conjoint |
| Attendance of religious services | 6 (More than once a week, Once a week, Once or twice a month, A few times a year, Seldom, Never) | ISSP, Twin-2K |
| Political ideology | 8 (Very liberal, Liberal, Somewhat liberal, ..., Other) | ISSP, Twin-2K, Conjoint |
| Voted in the last general election | 2 (Yes, No) | ISSP |
| Place of living | 5 (A big city, The suburbs or outskirts of a big city, A town or a small city, A country village, A farm or home in the country) | ISSP |
| Marital status | 6 (Married, Living with a partner,